In [1]:
import random
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.options.display.max_columns = 100

import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import seaborn as sns
from collections import Counter
import json

import all_blocks
from merge_contracts import ContractsMerger
from generate_test import generate_preds
from get_distribution_utils import get_disrib_sums, cost_columns_to_datetime


c:\users\никита\appdata\local\programs\python\python39\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Подгружаем и обрабатываем датку

In [2]:
pays_df1 = pd.read_excel("data/Счета на оплату 3800-2023.XLSX")
pays_df2 = pd.read_excel("data/Счета на оплату 4200-4000-3800-2024.XLSX")
pays_df3 = pd.read_excel("data/Счета на оплату 5400-2023.XLSX")
pays_df4 = pd.read_excel("data/Счета на оплату 5400-2024.XLSX")
pays_df5 = pd.read_excel("data/Счета на оплату 5500-2023.XLSX")

In [3]:
pays_df = pd.concat([pays_df1, pays_df2, pays_df3, pays_df4, pays_df5], axis=0)

In [4]:
merger_df = pd.read_excel("data/Связь договор - здания.XLSX")
main_costs_df = pd.read_excel("data/Основные средства.XLSX")
squares = pd.read_excel("data/Площади зданий.XLSX")
serv_codes = pd.read_excel("data/Коды услуг.XLSX")

In [5]:
main_costs_df["тест_фича_нум"] = [random.randint(0, 100) for i in range(len(main_costs_df))]
main_costs_df["тест_фича_бул"] = [random.randint(0, 1) for i in range(len(main_costs_df))]
main_costs_df["тест_фича_дата"] = ["19.04.2006"] * len(main_costs_df)

In [6]:
merger = ContractsMerger(pays_df, merger_df, main_costs_df, squares, serv_codes)

E:\DIR_python_projects\ledaer_digital_transformation_24\zakupai\services\api\ml\lib\merge_contracts.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.needed_pays_df["time"] = self.needed_pays_df["Год"].map(lambda x: datetime(year=x + 1, month=1, day=1))


In [7]:
import time

t = time.time()
res, features = merger.start_merging()
print(time.time() - t)

68.52783608436584


In [8]:
# сохраняем результат с 1 страницы загрузки данных
res.to_csv("res_datetimes.csv")
with open("features.pkl", "wb") as f:
    pickle.dump(features, f)

res остается лежать на бэке где-то
features нужны на фронте чтобы блоки с признаками создать и на бэке дальше
то есть там всегда будут 3 блока условия + блоки действий + feature из этого файла

## Подгружаем граф, запускаем распределение

In [21]:
features = pickle.load(open("features.pkl", "rb"))
res = pd.read_csv("res_datetimes.csv", index_col=0)

C:\Users\843E~1\AppData\Local\Temp/ipykernel_22092/930177580.py:2: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  res = pd.read_csv("res_datetimes.csv", index_col=0)


In [22]:
numeric_features = [i[0] for i in features if i[2] != "date"]
date_features = [i[0] for i in features if i[2] == "date"]

res = cost_columns_to_datetime(res, date_features)

In [23]:
res_by_prime = res.groupby("prime")

In [50]:
raw_graph = json.load(open("graphs/graph (5).json", "r"))

In [52]:
raw_graph["nodes"][2]["type"] = "use_feature_distribution"

In [56]:
all_blocks.init_graph(raw_graph, features)

In [57]:
unique_primes = res["prime"].unique()
distrib_sums = get_disrib_sums(res_by_prime, res["prime"].unique(), numeric_features)

100%|██████████| 4717/4717 [00:09<00:00, 491.68it/s]


,Компания,Год,Номер счета,Позиция счета,ID услуги,ID договора,Дата отражения счета в учетной системе,Стоимость без НДС,prime,time,ID здания,Отношение действ. с,Отношение действ. до,Здание,Начало владения,Конец владения,Измер. действит. по,Измерение действ. с,Площадь,Единица измерения,ID основного средства,Класс основного средства,"Признак ""Используется в основной деятельности""","Признак ""Способ использования""",Площадь ОС,ЕИ площади,Дата начала действия связи с зданием,Дата окончания действия связи с зданием,Дата ввода в эксплуатацию,Дата выбытия,тест_фича_нум,тест_фича_бул,тест_фича_дата
0,3800,2023,5006170938,1,800003299,ДПН 3800/74661,2023-01-16 00:00:00,1135426.7,5006170938_1,2024-01-01 00:00:00,ЗДН 3800/1/265,2023-01-11 00:00:00,9999-12-31 00:00:00,ЗДН 3800/1/265,2007-11-26,2040-01-01,9999-12-31 00:00:00,NaN,616.3,М2,38006040010385572,60401018,0,1,1.1,М2,1900-01-01,2040-01-01,2007-11-26,NaN,77,0,2006-04-19
1,3800,2023,5006170938,1,800003299,ДПН 3800/74661,2023-01-16 00:00:00,1135426.7,5006170938_1,2024-01-01 00:00:00,ЗДН 3800/1/1538,2023-01-11 00:00:00,9999-12-31 00:00:00,ЗДН 3800/1/1538,2018-08-10,2040-01-01,9999-12-31 00:00:00,2018-08-10 00:00:00,430.0,М2,38006080400345271,60804001,0,0,0.0,М2,1900-01-01,2040-01-01,2020-03-01,NaN,64,1,2006-04-19


In [98]:
pays_by_prime = pays_df.groupby("prime")
res_by_prime = res.groupby("prime")
group_pay = pays_by_prime.get_group("5006170938_1")
group_res = res_by_prime.get_group("5006170938_1")

In [107]:
group_res = group_res.rename({"Год": "Год счета", "Дата отражения счета в учетной системе": "Дата отражения в учетной системе",
					  "ID услуги": "Услуга", "ID здания": "Здание", "Класс основного средства": "Класс ОС",
					  'Признак "Используется в основной деятельности"': "Признак использования в основной деятель",
					  'Признак "Способ использования"': "Признак передачи в аренду", "Площадь ОС": "Площадь"}, axis=1)

In [108]:
group_pay

,Компания,Год,Номер счета,Позиция счета,ID услуги,ID договора,Дата отражения счета в учетной системе,Стоимость без НДС,prime
351,3800,2023,5006170938,1,800003299,ДПН 3800/74661,2023-01-16 00:00:00,1135426.7,5006170938_1


In [109]:
group_res

,Компания,Год счета,Номер счета,Позиция счета,Услуга,ID договора,Дата отражения в учетной системе,Стоимость без НДС,prime,time,Здание,Отношение действ. с,Отношение действ. до,Здание,Начало владения,Конец владения,Измер. действит. по,Измерение действ. с,Площадь,Единица измерения,ID основного средства,Класс ОС,Признак использования в основной деятель,Признак передачи в аренду,Площадь,ЕИ площади,Дата начала действия связи с зданием,Дата окончания действия связи с зданием,Дата ввода в эксплуатацию,Дата выбытия,тест_фича_нум,тест_фича_бул,тест_фича_дата
0,3800,2023,5006170938,1,800003299,ДПН 3800/74661,2023-01-16 00:00:00,1135426.7,5006170938_1,2024-01-01 00:00:00,ЗДН 3800/1/265,2023-01-11 00:00:00,9999-12-31 00:00:00,ЗДН 3800/1/265,2007-11-26,2040-01-01,9999-12-31 00:00:00,NaN,616.3,М2,38006040010385572,60401018,0,1,1.1,М2,1900-01-01,2040-01-01,2007-11-26,NaN,77,0,2006-04-19
1,3800,2023,5006170938,1,800003299,ДПН 3800/74661,2023-01-16 00:00:00,1135426.7,5006170938_1,2024-01-01 00:00:00,ЗДН 3800/1/1538,2023-01-11 00:00:00,9999-12-31 00:00:00,ЗДН 3800/1/1538,2018-08-10,2040-01-01,9999-12-31 00:00:00,2018-08-10 00:00:00,430.0,М2,38006080400345271,60804001,0,0,0.0,М2,1900-01-01,2040-01-01,2020-03-01,NaN,64,1,2006-04-19


In [110]:
answer_ex = pd.read_csv("data/Шаблон ответа.csv", sep=";")

In [111]:
for i in answer_ex.columns:
	if i not in group_res.columns:
		print("add ", i)

for i in group_res.columns:
	if i not in answer_ex.columns:
		print("del ", i)

add  Номер позиции распределения
add  ID услуги
add  Сумма распределения
add  Счет главной книги
del  Стоимость без НДС
del  prime
del  time
del  Отношение действ. с
del  Отношение действ. до
del  Начало владения
del  Конец владения
del  Измер. действит. по
del  Измерение действ. с
del  Единица измерения
del  ЕИ площади
del  Дата начала действия связи с зданием
del  Дата окончания действия связи с зданием
del  Дата ввода в эксплуатацию
del  Дата выбытия
del  тест_фича_нум
del  тест_фича_бул
del  тест_фича_дата


In [89]:
answer_ex

,Компания,Год счета,Номер счета,Позиция счета,Номер позиции распределения,Дата отражения в учетной системе,ID договора,Услуга,ID услуги,Здание,Класс ОС,ID основного средства,Признак использования в основной деятель,Признак передачи в аренду,Площадь,Сумма распределения,Счет главной книги
0,5500,2024,5006666579,1,1,NaN,ДПН 5500/183855,800001855,S004,ЗДН 5500/1/1064,60401018,55006040015814050,X,NaN,"44,6",1604.71,7048209010


In [ ]:


def generate_preds_new(pays_df, serv_codes, preds_df, distrib_sums):
	"""
	Функция для генерации таблицы с предиктами распределенных сумм.
	Принимает на вход:
		- pays_df: pd.DataFrame, счета на оплату (сконкатенированные в один DF), изначальный набор счетов;
		- serv_codes: pd.DataFrame, коды услуг;
		- preds_df: pd.DataFrame, смёрдженная таблица, полученная через merge_contracts;
		- distrib_sums: defaultDict, словарь с распределенной суммой для каждого номера счета + позиции счета.

	Возвращает pd.DataFrame со сгенерированной распределенной на каждую позицию счета суммой.
	"""
	serv_codes = {r["ID услуги"]: r["Класс услуги"] for _, r in serv_codes.iterrows()}

	pays_df["prime"] = pays_df["Номер счета"].astype(str) + "_" + pays_df["Позиция счета"].astype(str)
	pays_by_prime = pays_df.groupby("prime")		# тут на каждый prime key одна строка
	res_by_prime = preds_df.groupby("prime")

	res_df = []
	j = 0
	for prime, dist in tqdm(pays_by_prime):
		prime_group = pays_by_prime.get_group(prime)

		j += 1
		try:
			data = res_by_prime.get_group(prime)
		except:
			print("WARNING! prime_id not in res_by_prime df")
			continue

		d1 = pd.concat([prime_group.iloc[0], data.iloc[0]], axis=0)
		sums = distrib_sums[prime]
		for i, s in enumerate(sums):
			d = d1.copy()
			d["Номер позиции распределения"] = i+1
			d["Сумма распределения"] = s
			d.rename({"Год": "Год счета", "Дата отражения счета в учетной системе": "Дата отражения в учетной системе",
					  "ID услуги": "Услуга", "ID здания": "Здание", "Класс основного средства": "Класс ОС",
					  'Признак "Используется в основной деятельности"': "Признак использования в основной деятель",
					  'Признак "Способ использования"': "Признак передачи в аренду", "Площадь ОС": "Площадь"}, inplace=True)

			d = d[['Компания', 'Год счета', 'Номер счета', 'Позиция счета',
			   'Номер позиции распределения', 'Дата отражения в учетной системе',
			   'ID договора', 'Услуга', 'Здание', 'Класс ОС',
			   'ID основного средства', 'Признак использования в основной деятель',
			   'Признак передачи в аренду', 'Площадь', 'Сумма распределения']]
			d = pd.DataFrame(d).transpose()
			res_df.append(d)
	res_df = pd.concat(res_df)

	res_df = res_df.loc[:,~res_df.columns.duplicated()].copy()
	res_df["Счет главной книги"] = 7048209010
	res_df["ID услуги"] = res_df["Услуга"].map(serv_codes)

	return res_df

In [73]:
preds = generate_preds_new(pays_df, serv_codes, res, distrib_sums)

100%|██████████| 124651/124651 [02:39<00:00, 780.06it/s] 


In [74]:
preds

,Компания,Год счета,Номер счета,Позиция счета,Номер позиции распределения,Дата отражения в учетной системе,ID договора,Услуга,Здание,Класс ОС,ID основного средства,Признак использования в основной деятель,Признак передачи в аренду,Площадь,Сумма распределения,Счет главной книги,ID услуги
0,3800,2023,5006170938,1,1,2023-01-16 00:00:00,ДПН 3800/74661,800003299,ЗДН 3800/1/265,60401018,38006040010385572,0,1,616.3,0.589028,7048209010,S001
0,3800,2023,5006170938,1,2,2023-01-16 00:00:00,ДПН 3800/74661,800003299,ЗДН 3800/1/265,60401018,38006040010385572,0,1,616.3,0.410972,7048209010,S001
0,5400,2023,5006176256,1,1,2023-01-18 00:00:00,ДПН 5400/143136,800002266,ЗДН 5400/1/7432,60804001,54006080400468090,1,0,150.9,1.0,7048209010,S036
0,5400,2023,5006176256,2,1,2023-01-18 00:00:00,ДПН 5400/143136,800002266,ЗДН 5400/1/7432,60804001,54006080400468090,1,0,150.9,1.0,7048209010,S036
0,5400,2023,5006197559,1,1,2023-02-01 00:00:00,ДПН 5400/143743,800006222,ЗДН 5400/1/6824,60804001,54006080400468690,1,0,971.09,1.0,7048209010,S012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,4200,2024,5006861096,1,83,2024-05-28 00:00:00,ДПН 4200/207590,800007377,ЗДН 4200/1/3960,60804001,42006080400466500,1,0,236.7,0.000041,7048209010,S004
0,4200,2024,5006861096,1,84,2024-05-28 00:00:00,ДПН 4200/207590,800007377,ЗДН 4200/1/3960,60804001,42006080400466500,1,0,236.7,0.00023,7048209010,S004
0,4200,2024,5006861096,1,85,2024-05-28 00:00:00,ДПН 4200/207590,800007377,ЗДН 4200/1/3960,60804001,42006080400466500,1,0,236.7,0.000084,7048209010,S004
0,4200,2024,5006861117,1,1,2024-05-28 00:00:00,ДПН 4200/207349,800000706,ЗДН 4200/1/423,60401018,42006040015620131,0,0,13282.0,0.5,7048209010,S001


In [17]:
preds.to_csv("Результат_распределенные_счета.csv")